### Pre-processing par seuil bof: For HSV, hue range is [0,179], saturation range is [0,255], and value range is [0,255]

In [ ]:
from scipy.signal import find_peaks

def annotate_peaks(ax, hist, color, height=0.01):
    peaks, _ = find_peaks(hist, height=height)
    for p in peaks:
        ax.annotate(f"{int(p)}", xy=(p, hist[p]), xytext=(p, hist[p] + max(hist)*0.02),
                    textcoords='data', ha='center', fontsize=8, color=color, rotation=45)

# Dossier des images
img_dir = "dataset_project_iapr2025/train/"
img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")])

# Itérer sur toutes les images
for filename in img_files:
    print(f"Traitement de : {filename}")
    path = os.path.join(img_dir, filename)

    # === Ton pipeline (inchangé) ===
    img = cv.imread(path)
    img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    # Masquage noir
    img_noir = img.copy()
    img_noir_rgb = cv.cvtColor(img_noir, cv.COLOR_BGR2RGB)

    # mask BACKGROUND
    mask_rgb = (
        (img_noir_rgb[:, :, 0] >= 105) & (img_noir_rgb[:, :, 0] <= 195) &
        (img_noir_rgb[:, :, 1] >= 105) & (img_noir_rgb[:, :, 1] <= 195) &
        (img_noir_rgb[:, :, 2] >= 105) & (img_noir_rgb[:, :, 2] <= 195)
    )

    img_noir_hsv = cv.cvtColor(img_noir_rgb, cv.COLOR_RGB2HSV)

    mask_hsv = (
        (((img_noir_hsv[:, :, 0] >= 110) & (img_noir_hsv[:, :, 0] <= 120)) |
        ((img_noir_hsv[:, :, 1] >= 0) & (img_noir_hsv[:, :, 1] <= 50))) |
        ((img_noir_hsv[:, :, 2] >= 120) & (img_noir_hsv[:, :, 2] <= 180))
    )

    mask_total = mask_rgb & mask_hsv
    img_noir_rgb[mask_total] = [0, 0, 0]
    mask_choco = np.where(mask_total, 0, 255).astype(np.uint8)

    # Histogrammes
    img_hsv = cv.cvtColor(img_rgb, cv.COLOR_RGB2HSV)
    r_hist = cv.calcHist([img_rgb], [0], mask_choco, [256], [0, 256]).flatten()
    g_hist = cv.calcHist([img_rgb], [1], mask_choco, [256], [0, 256]).flatten()
    b_hist = cv.calcHist([img_rgb], [2], mask_choco, [256], [0, 256]).flatten()
    h_hist = cv.calcHist([img_hsv], [0], mask_choco, [180], [0, 180]).flatten()
    s_hist = cv.calcHist([img_hsv], [1], mask_choco, [256], [0, 256]).flatten()
    v_hist = cv.calcHist([img_hsv], [2], mask_choco, [256], [0, 256]).flatten()

    # Figure 1
    fig, axs = plt.subplots(2, 2, figsize=(14, 14))
    axs[0, 0].imshow(img_rgb)
    axs[0, 0].set_title(f"Image originale - {filename}")
    axs[0, 0].axis("off")

    axs[0, 1].imshow(img_noir_rgb)
    axs[0, 1].set_title("Image après correction noire")
    axs[0, 1].axis("off")

    axs[1, 0].plot(r_hist, color='r', label='R')
    axs[1, 0].plot(g_hist, color='g', label='G')
    axs[1, 0].plot(b_hist, color='b', label='B')
    axs[1, 0].set_title("Histogrammes RGB du mask")
    axs[1, 0].legend()
    annotate_peaks(axs[1, 0], r_hist, color='r')
    annotate_peaks(axs[1, 0], g_hist, color='g')
    annotate_peaks(axs[1, 0], b_hist, color='b')

    axs[1, 1].plot(h_hist, color='purple', label='H')
    axs[1, 1].plot(s_hist, color='orange', label='S')
    axs[1, 1].plot(v_hist, color='gray', label='V')
    axs[1, 1].set_title("Histogrammes HSV du mask")
    axs[1, 1].legend()
    annotate_peaks(axs[1, 1], h_hist, color='purple')
    annotate_peaks(axs[1, 1], s_hist, color='orange')
    annotate_peaks(axs[1, 1], v_hist, color='gray')

    plt.tight_layout()
    plt.show()


### Find objects on blank background :

In [ ]:
def apply_closing(mask, size):
    return closing(mask > 0, square(size))
def apply_opening(mask, size):
    return opening(mask > 0, square(size))

def remove_objects(binary_mask, min_size):
    return remove_small_objects(binary_mask > 0, min_size=min_size)

def apply_gaussian_several_times(img, kernel_size, sigma, times):
    copy = img.copy()
    for _ in range(times):
        copy = cv.GaussianBlur(copy, (kernel_size, kernel_size), sigma)
    return copy


def detect_contours_single_img_debug(img, size=11):
    img = cv.resize(img, (0, 0), fx=0.25, fy=0.25)  # Réduction à 25% de la taille originale
    img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    img_copy = img_rgb.copy()

    remove_objects_size = 250 # 200
    thres2 = 2
    sigma = 50 #16 #2.7, 12 bien
    p2 = 32
    th_sigma = 5#2.5
    close_th = 30

    # Atténuation des canaux rouge et vert
    img_att = img_rgb.copy()
    img_att[:, :, 0] = img_att[:, :, 0] * 1  # red
    img_att[:, :, 1] = img_att[:, :, 1] * 1  # green
    img_att[:, :, 2] = img_att[:, :, 2] * 1  # blue

    img_hsv = cv.cvtColor(img_att, cv.COLOR_RGB2HSV)
    img_hsv[:, :, 0] = img_hsv[:, :, 0] * 1  # Atténuation de la teinte
    img_hsv[:, :, 1] = img_hsv[:, :, 1] * 0.5 # Atténuation de la saturation
    img_hsv[:, :, 2] = img_hsv[:, :, 2] * 1  # Atténuation de la valeur (luminosité)
    img_att = cv.cvtColor(img_hsv, cv.COLOR_HSV2RGB)

    # Inversion + passage en niveaux de gris
    imgGray = cv.cvtColor(255 - img_att, cv.COLOR_RGB2GRAY)

    low_pass = apply_gaussian_several_times(imgGray, size, sigma, 10)

    
    high_pass = cv.subtract(imgGray, low_pass)
    _, thresholded = cv.threshold(high_pass, thres2, 255, cv.THRESH_BINARY)
    thresholded = np.uint8(remove_objects(thresholded, remove_objects_size)) * 255
    thresholded_opening = np.uint8(apply_opening(thresholded, 2)) * 255
    thresholded_closing = np.uint8(apply_closing(thresholded_opening, close_th)) * 255
    thresholded_blur = cv.GaussianBlur(thresholded_closing, (size, size), th_sigma)

    contours, _ = cv.findContours(thresholded_blur, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        area = cv.contourArea(cnt)
        if area > 500:  # adapte ce seuil si nécessaire
            cv.drawContours(img_copy, [cnt], -1, (255, 0, 0), 3)
    # Retourne aussi l'image atténuée
    return img_rgb, img_att, imgGray, low_pass, high_pass, thresholded, thresholded_closing, thresholded_blur, img_copy

X = df_background_train[feature_cols]
y_pred = model.predict(X)

# === Récupérer les IDs prédits comme background 1 ===
ids_predicted_1 = df_background_train['id'][y_pred == 1].values

for image_id in ids_predicted_1:
    filename = f"L{image_id}.JPG"
    path = os.path.join(img_train_dir, filename)


    if not os.path.exists(path):
        print(f"⚠️ Fichier manquant : {filename}")
        continue

    img = cv.imread(path)

    imgs = detect_contours_single_img_debug(img)

    titles = [
    "Image originale",
    "Image atténuée",
    "Grayscale inversé",
    "Low-pass filter",
    "High-pass",
    "Thresholded",
    "Closing",
    "Gaussian blur",
    "Contours détectés"
    ]

    plt.figure(figsize=(22, 12))
    for i, (title, im) in enumerate(zip(titles, imgs)):
        plt.subplot(3, 3, i + 1)
        cmap = "gray" if len(im.shape) == 2 else None
        plt.imshow(im, cmap=cmap)
        plt.title(title, fontsize=10)
        plt.axis("off")

    plt.suptitle(f"Étapes de traitement - {filename}", fontsize=14)
    plt.tight_layout()
    plt.show()



### Version avec watershed:

In [ ]:
def apply_watershed_segmentation(binary_mask, img_rgb, border_thickness=2):
    binary = binary_mask.copy()
    if binary.dtype != np.uint8:
        binary = np.uint8(binary)

    # ✅ 1. Transformée de distance pour un meilleur foreground
    dist_transform = cv.distanceTransform(binary, cv.DIST_L2, 5)
    _, sure_fg = cv.threshold(dist_transform, 0.2 * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)

    # ✅ 2. Dilatation légère pour le background
    sure_bg = cv.dilate(binary, np.ones((3, 3), np.uint8), iterations=1)

    # ✅ 3. Zone inconnue
    unknown = cv.subtract(sure_bg, sure_fg)

    # ✅ 4. Création des marqueurs
    _, markers = cv.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    # ✅ 5. Application du watershed
    img_copy = img_rgb.copy()
    markers = cv.watershed(img_copy, markers)

    # ✅ 6. Création des contours
    contour_mask = (markers == -1).astype(np.uint8) * 255

    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (border_thickness, border_thickness))
    thick_contour = cv.dilate(contour_mask, kernel, iterations=2)

    # ✅ 7. Application des bords en rouge
    segmented = img_copy.copy()
    segmented[thick_contour == 255] = [255, 0, 0]

    return segmented, markers




def apply_closing(mask, size):
    return closing(mask > 0, square(size))
def apply_opening(mask, size):
    return opening(mask > 0, square(size))

def remove_objects(binary_mask, min_size):
    return remove_small_objects(binary_mask > 0, min_size=min_size)

def apply_gaussian_several_times(img, kernel_size, sigma, times):
    copy = img.copy()
    for _ in range(times):
        copy = cv.GaussianBlur(copy, (kernel_size, kernel_size), sigma)
    return copy


def detect_contours_single_img_debug(img, size=11):
    img = cv.resize(img, (0, 0), fx=0.25, fy=0.25)  # Réduction à 25% de la taille originale
    img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    img_copy = img_rgb.copy()

    remove_objects_size = 250 # 200
    thres2 = 2
    sigma = 50 #16 #2.7, 12 bien
    p2 = 32
    th_sigma = 5#2.5
    close_th = 30

    # Atténuation des canaux rouge et vert
    img_att = img_rgb.copy()
    img_att[:, :, 0] = img_att[:, :, 0] * 1  # red
    img_att[:, :, 1] = img_att[:, :, 1] * 1  # green
    img_att[:, :, 2] = img_att[:, :, 2] * 1  # blue

    img_hsv = cv.cvtColor(img_att, cv.COLOR_RGB2HSV)
    img_hsv[:, :, 0] = img_hsv[:, :, 0] * 1  # Atténuation de la teinte
    img_hsv[:, :, 1] = img_hsv[:, :, 1] * 0.5 # Atténuation de la saturation
    img_hsv[:, :, 2] = img_hsv[:, :, 2] * 1  # Atténuation de la valeur (luminosité)
    img_att = cv.cvtColor(img_hsv, cv.COLOR_HSV2RGB)

    # Inversion + passage en niveaux de gris
    imgGray = cv.cvtColor(255 - img_att, cv.COLOR_RGB2GRAY)

    low_pass = apply_gaussian_several_times(imgGray, size, sigma, 10)

    
    high_pass = cv.subtract(imgGray, low_pass)
    _, thresholded = cv.threshold(high_pass, thres2, 255, cv.THRESH_BINARY)
    thresholded = np.uint8(remove_objects(thresholded, remove_objects_size)) * 255
    thresholded_opening = np.uint8(apply_opening(thresholded, 2)) * 255
    thresholded_closing = np.uint8(apply_closing(thresholded_opening, close_th)) * 255
    thresholded_blur = cv.GaussianBlur(thresholded_closing, (size, size), th_sigma)

    contours, _ = cv.findContours(thresholded_blur, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        area = cv.contourArea(cnt)
        if area > 500:  # adapte ce seuil si nécessaire
            cv.drawContours(img_copy, [cnt], -1, (255, 0, 0), 3)
    # Retourne aussi l'image atténuée
    return img_rgb, img_att, imgGray, low_pass, high_pass, thresholded, thresholded_closing, thresholded_blur, img_copy

# === Pipeline principal ===
X = df_background_train[feature_cols]
y_pred = model.predict(X)
ids_predicted_1 = df_background_train['id'][y_pred == 1].values

for image_id in ids_predicted_1:
    filename = f"L{image_id}.JPG"
    path = os.path.join(img_train_dir, filename)

    if not os.path.exists(path):
        print(f"⚠️ Fichier manquant : {filename}")
        continue

    img = cv.imread(path)

    imgs = detect_contours_single_img_debug(img)
    img_rgb, img_att, imgGray, low_pass, high_pass, thresholded, thresholded_closing, thresholded_blur, img_copy = imgs

    # Appliquer watershed à partir de l’image originale et du masque thresholded_closing
    segmented_img, _ = apply_watershed_segmentation(thresholded_closing, img_rgb)

    titles = ["Image originale", "thresholded_closing", "Contours détectés", "Segmentation Watershed"]
    images = [imgs[0], imgs[6], imgs[-1], segmented_img]  # img_rgb, thresholded_closing, img_copy, segmented_img

    plt.figure(figsize=(18, 6))
    for i, (title, im) in enumerate(zip(titles, images)):
        plt.subplot(1, 4, i + 1)
        cmap = "gray" if len(im.shape) == 2 else None
        plt.imshow(im, cmap=cmap)
        plt.title(title, fontsize=12)
        plt.axis("off")

    plt.suptitle(f"Résultats - {filename}", fontsize=14)
    plt.tight_layout()
    plt.show()

